# Contour CSVs: stitching, RANSAC and registration error from pre-analysed contour centres

Same analysis as `stitch_viewer.ipynb`, but the contact centres come from **pre-analysed contour
CSV files** instead of the SEM images and `.oas` files: one CSV per image, one row per contact,
columns `DesignX_Add`, `DesignY_Add` (design centre) and `SEMX_Add`, `SEMY_Add` (SEM centre), in
**global mask coordinates, nm**. Design and SEM on the same row are the same contact, so they are
not matched again.

1. **Read** every `*.csv` in `CSV_DIR` (`read_contour_folder`). Rows with a missing value are left
   out and counted; files without any contact are listed and left out.
2. **Stitching**, translation per tile, as for the images: the SEM centres (`stitch_tiles`) and,
   the same way, the design centres, which can be offset between files. There is no metadata
   CSV, so each tile's box is the bounding box of its design centres, grown by `TILE_MARGIN_NM`
   (`tile_boxes_from_points`): two tiles overlap where both report contacts. A viewer shows the
   centres, tile boxes and overlaps (`PointsStitchView`).
3. **Errors** per placement (`paired_errors`, raw SEM − design, no affine), then one entry per
   physical contact (`merge_observations`).
   **In-image correction** (2b, 3b): the distortion shared by all images, measured against the
   design (`estimate_map`) and removed from every SEM point before stitching again; checked with the
   overlaps (`overlap_slopes`).
4. **RANSAC** global affine, **moving-window** affine and **row pitch** on the merged contacts;
   RANSAC and moving window on the in-image corrected contacts (section 8); the **drift curve** along
   y removed before RANSAC (section 9); the **drift per stripe** in measurement order, removed
   before stitching everything again (section 10); the **stitching residuals** of every input,
   with translation and with translation + rotation per tile (section 11); RANSAC and moving
   window on both corrections, placed with translation + rotation per tile (section 11b).
5. **Analysis window** (`AnalysisWindow`, section 12): stitching inputs (the section 2 viewer),
   RANSAC monitor, registration error (RANSAC and moving window, each with the mean error per row
   **without affine** as a reference), moving-window tuner, row pitch. The corrected results are
   extra lines in these plots and have one tab each.

The windows block this notebook until they are closed. Positions in µm, errors in nm.

In [ ]:
CSV_DIR = r"path\to\csv_folder"
TILE_MARGIN_NM = 25.0       # tile box = bounding box of its design centres grown by this; keeps neighbours that share one row of contacts
MAX_MATCH_NM = 25.0         # SEM overlaps: pair contacts closer than this, the largest B - A stage offset (< pitch / 2)
DESIGN_MAX_MATCH_NM = 25.0  # design overlaps: the largest offset between neighbouring files (< pitch / 2)
OUTLIER_FACTOR = 3.0        # overlap outlier if residual > this x the median residual
MIN_MATCHED = 5             # pairs with fewer matched contacts are not used (reported below)
GROUP_RADIUS_NM = 5.0       # observations from different tiles whose stitched design positions are this close = one contact
SPREAD_FLAG_NM = 2.0        # flag merged contacts whose SEM observations differ by more than this
USE_OPENGL = True           # set False if the viewer window stays black
RANSAC_PLACEMENT = "mean"   # RANSAC on the merged contacts of this placement: "mean" (stitched) or "first"
RANSAC_THRESHOLD_NM = 0.5   # RANSAC inlier if |G(SEM) - design| < this
RANSAC_SEED = 0             # RANSAC random samples are reproducible for a given seed
RANSAC_DELAY_MS = 300       # RANSAC monitor: pause between steps
ROW_GAP_NM = 10.0           # registration error per row: a new row starts where sorted design y jumps by more than this
MOVING_WINDOW_UM = 40.0     # moving-window affine: window height along y
MOVING_STEP_UM = 5.0        # moving-window affine: the window moves up this much each step
INTRAFIELD_NODES = 9        # in-image distortion map: nodes per axis over the image, from edge to edge
DRIFT_WINDOW_UM = 40.0      # drift curve: height of the local straight-line fit along y
FIRST_STRIPE_UPWARD = True  # serpentine scan: the first stripe (smallest x) was measured from its bottom up, then alternately

In [ ]:
import numpy as np
import pandas as pd
import pyqtgraph as pg

from affine_ransac.fitting.drift import drift_curve, stripe_drift
from affine_ransac.fitting.moving_window import moving_window_affine
from affine_ransac.fitting.ransac import ransac_affine
from affine_ransac.geometry.affine import apply_affine, report_terms
from affine_ransac.intrafield import correct_points, estimate_map, overlap_slopes
from affine_ransac.io.contour_csv import read_contour_folder
from affine_ransac.pipeline import (measurement_pairs, residual_summary, stitch_tiles, stitch_tiles_rigid,
                                    tie_residuals, tile_boxes_from_points, tile_stripes)
from affine_ransac.registration import error_summary, merge_observations, paired_errors, row_means, row_pitch
from affine_ransac.stitching import placement_corrections
from affine_ransac.view_analysis import AnalysisWindow
from affine_ransac.view_drift import StripeDriftView
from affine_ransac.view_stitch_residuals import StitchResidualView
from affine_ransac.view_points import PointsStitchView

## 1. Read the contour CSVs

One row per file: contacts used, rows left out (missing value), the area covered by its design
centres (µm) and its mean SEM − design before stitching (nm).

In [ ]:
all_tiles = read_contour_folder(CSV_DIR)
empty = [t.name for t in all_tiles if len(t.design_nm) == 0]
if empty:
    print(f"!! {len(empty)} file(s) without any contact, left out: {empty}")
tiles = [t for t in all_tiles if len(t.design_nm)]
tile_ids = [t.name for t in tiles]
print(f"{len(tiles)} of {len(all_tiles)} files used, {sum(len(t.design_nm) for t in tiles)} contacts, "
      f"{sum(t.dropped for t in all_tiles)} rows left out (missing value)")

pd.DataFrame({
    "tile": tile_ids,
    "contacts": [len(t.design_nm) for t in tiles],
    "rows_left_out": [t.dropped for t in tiles],
    "x_min_um": [t.design_nm[:, 0].min() / 1000 for t in tiles], "x_max_um": [t.design_nm[:, 0].max() / 1000 for t in tiles],
    "y_min_um": [t.design_nm[:, 1].min() / 1000 for t in tiles], "y_max_um": [t.design_nm[:, 1].max() / 1000 for t in tiles],
    "mean_dx_nm": [(t.sem_nm - t.design_nm)[:, 0].mean() for t in tiles],
    "mean_dy_nm": [(t.sem_nm - t.design_nm)[:, 1].mean() for t in tiles],
}).round(4)

## 2. Stitching: SEM centres and design centres

The second cell opens a viewer of what the stitching used (closes with the window): design (blue)
and SEM (red) centres at the CSV coordinates, the tile boxes (grey), the overlaps used (yellow),
skipped (dashed grey: fewer than `MIN_MATCHED` contacts matched) and not matched (dashed magenta,
SEM; dash-dot orange, design), and the tiles that could not be stitched (thick magenta). Click a
legend entry to hide or show a layer; *Tile names* labels each box with its file name.

In [ ]:
sem_points = [t.sem_nm for t in tiles]
design_points = [t.design_nm for t in tiles]
centers, sizes = tile_boxes_from_points(design_points, TILE_MARGIN_NM)  # bounding box of the design centres, grown
stitch = stitch_tiles(sem_points, centers, sizes, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
design_stitch = stitch_tiles(design_points, centers, sizes, DESIGN_MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED,
                             label="design")

for name, result in (("SEM", stitch), ("Design", design_stitch)):
    failed = [r for r in result.rejected if r.failed]
    print(f"{name}: {len(result.pairs)} overlapping pairs used, {len(failed)} not matched, "
          f"{len(result.rejected) - len(failed)} skipped (fewer than {MIN_MATCHED} contacts in the overlap); "
          f"corrections |x| max {np.nanmax(np.abs(result.corrections[:, 0])):.2f} nm, "
          f"|y| max {np.nanmax(np.abs(result.corrections[:, 1])):.2f} nm")
    if failed or len(result.unplaced):
        print(f"!! {name.upper()} STITCHING INCOMPLETE. Tiles not stitched: {[tile_ids[k] for k in result.unplaced]}")
        display(pd.DataFrame([{"tile_a": tile_ids[r.i], "tile_b": tile_ids[r.j], "contacts_in_overlap": r.in_box,
                               "matched": r.matched} for r in failed]))

pd.DataFrame({
    "tile": tile_ids,
    "sem_correction_x_nm": stitch.corrections[:, 0], "sem_correction_y_nm": stitch.corrections[:, 1],
    "design_correction_x_nm": design_stitch.corrections[:, 0], "design_correction_y_nm": design_stitch.corrections[:, 1],
}).round(2)

In [ ]:
app = pg.mkQApp("Stitching inputs")
points_view = PointsStitchView(tile_ids, sem_points, design_points, centers, sizes, stitch, design_stitch,
                               use_opengl=USE_OPENGL)
points_view.show()
app.exec()

### 2b. In-image distortion: one map for all images, measured against the design

Every image can be distorted the same way inside (e.g. the SEM scan is not perfectly linear).
`estimate_map` measures it from the design: per image, SEM − design after that image's own affine
(so only what no affine describes), on `INTRAFIELD_NODES` × `INTRAFIELD_NODES` nodes from edge to edge
of the image, from all images together (images with too few contacts are left out). Positions in the
image are relative to its centre, from the CSV's `DesignX`, `DesignY` columns. Every SEM point is then
corrected (`correct_points`) and the corrected points are stitched again (translation, as above).

Mask errors at fixed places on the mask fall at a different place in every image and average out;
mask errors that repeat at the image spacing (e.g. a writer field of the same size) would be taken for
SEM distortion. The second cell checks:
- **Overlaps** (`overlap_slopes`, no design involved: the same contact measured in two images, so mask
  errors cancel): how B − A changes across the overlap, median over the vertical and the horizontal
  overlaps (ppm). A distortion shared by all images gives the same slope in every overlap; after a
  correct map the slopes are ~0.
- **Halves of the field**: the map from the lower and from the upper half of the images; a distortion
  of the SEM is the same in both.

In [ ]:
image_centers = np.array([t.center_nm for t in tiles])  # global - local (DesignX, DesignY) position
if np.isnan(image_centers).any():
    raise RuntimeError("The in-image map needs the image centres: these CSVs have no DesignX, DesignY columns")
design_local = [d - c for d, c in zip(design_points, image_centers)]
sem_local = [s - c for s, c in zip(sem_points, image_centers)]
intrafield = estimate_map(design_local, sem_local, INTRAFIELD_NODES)
print(f"Map from {intrafield.images} of {len(tiles)} images, up to {np.abs(intrafield.shift_nm).max():.3f} nm")
for axis, name in enumerate(("dx", "dy")):
    print(f"{name} (nm) at the nodes: rows from the top of the image (y, nm), columns from the left (x, nm)")
    display(pd.DataFrame(intrafield.shift_nm[::-1, :, axis], index=np.round(intrafield.y_nm[::-1]),
                         columns=np.round(intrafield.x_nm)).round(3))

ic_points = [correct_points(p, c, intrafield) for p, c in zip(sem_points, image_centers)]
ic_stitch = stitch_tiles(ic_points, centers, sizes, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
print(f"Corrected points stitched again: {len(ic_stitch.pairs)} pairs used, {len(ic_stitch.unplaced)} tile(s) not stitched")

In [ ]:
slope_names = {(0, 0): "d(dx)/dx", (1, 1): "d(dy)/dy", (0, 1): "d(dx)/dy", (1, 0): "d(dy)/dx"}
check = {}
for name, points, result in (("translation", sem_points, stitch), ("in-image corrected", ic_points, ic_stitch)):
    slopes = overlap_slopes(points, result, centers)
    check[name] = {f"{kind} overlaps, {label} (ppm)": np.median(s[:, i, j])
                   for kind, s in slopes.items() for (i, j), label in slope_names.items()}
    ties = np.linalg.norm(tie_residuals(points, result, result.corrections), axis=1)
    check[name]["overlap ties RMS (nm)"] = np.sqrt((ties ** 2).mean())
display(pd.DataFrame(check).round(3))

lower = image_centers[:, 1] < np.median(image_centers[:, 1])
halves = [estimate_map([design_local[k] for k in np.flatnonzero(part)], [sem_local[k] for k in np.flatnonzero(part)],
                       INTRAFIELD_NODES) for part in (lower, ~lower)]
print(f"Map from the lower vs the upper half of the field: largest difference "
      f"{np.abs(halves[0].shift_nm - halves[1].shift_nm).max():.3f} nm")

## 3. Errors per placement (raw: SEM − design, no affine removed)

Each CSV row is one observation; a contact in an overlap is observed once per tile (2 in a strip,
up to 4 in a corner). `merge_observations` turns them into one entry per physical contact:
observations from different tiles whose **stitched design** positions are within
`GROUP_RADIUS_NM` are averaged, and their **spread** (largest distance between the SEM
observations) is kept. The table compares the placements: nominal (CSV coordinates as given),
stitched (corrections average 0), stitched with the first tile fixed.

In [ ]:
observations, merged = {}, {}
for placement, (sem_corrections, design_corrections) in placement_corrections(stitch.corrections,
                                                                               design_stitch.corrections).items():
    observations[placement] = paired_errors(sem_points, design_points, sem_corrections, design_corrections)
    merged[placement] = merge_observations(observations[placement], design_stitch.corrections, GROUP_RADIUS_NM)

for placement, e in observations.items():
    if e.skipped:
        print(f"!! {placement}: {len(e.skipped)} tile(s) NOT MEASURED (no error computed):")
        display(pd.DataFrame([{"tile": tile_ids[k], "reason": reason} for k, reason in e.skipped.items()]))

m = merged["mean"]
counts = pd.Series(m.count).value_counts().sort_index()
print(f"{len(observations['mean'].tile)} observations -> {len(m.count)} contacts; "
      f"observations per contact: {({int(k): int(v) for k, v in counts.items()})}")
overlap = m.count > 1
if overlap.any():
    print(f"Spread of merged overlap contacts (stitched): median {np.median(m.spread_nm[overlap]):.2f} nm, "
          f"max {m.spread_nm.max():.2f} nm; {(m.spread_nm > SPREAD_FLAG_NM).sum()} above {SPREAD_FLAG_NM} nm")
    worst = np.argsort(m.spread_nm)[::-1][:10]
    display(pd.DataFrame({
        "x_um": m.design_nm[worst, 0] / 1000, "y_um": m.design_nm[worst, 1] / 1000,
        "spread_nm": m.spread_nm[worst], "observations": m.count[worst],
        "tiles": [[tile_ids[k] for k in observations["mean"].tile[m.members[g]]] for g in worst],
    }).round(4))

pd.DataFrame({placement: error_summary(e.error_nm) for placement, e in merged.items()}).round(2)

### 3b. Errors with the in-image correction

The same errors (`paired_errors`, `merge_observations`) with the corrected SEM points and their
stitching (stitched placement); the design stitching is unchanged.

In [ ]:
ic_observations = paired_errors(ic_points, design_points, ic_stitch.corrections, design_stitch.corrections)
ic_contacts = merge_observations(ic_observations, design_stitch.corrections, GROUP_RADIUS_NM)
for name, m in (("translation (mean)", merged["mean"]), ("in-image corrected", ic_contacts)):
    overlap = m.count > 1
    if overlap.any():
        print(f"{name}: {len(m.count)} contacts; spread of merged overlap contacts median "
              f"{np.median(m.spread_nm[overlap]):.3f} nm, max {m.spread_nm.max():.3f} nm, "
              f"{(m.spread_nm > SPREAD_FLAG_NM).sum()} above {SPREAD_FLAG_NM} nm")
pd.DataFrame({"translation (mean)": error_summary(merged["mean"].error_nm),
              "in-image corrected": error_summary(ic_contacts.error_nm)}).round(2)

## 4. RANSAC global affine (on the merged contacts)

`ransac_affine` finds one affine G for the whole field that maps the stitched SEM centres onto the
design centres: random 3-point samples, the model with the most inliers (residual <
`RANSAC_THRESHOLD_NM`), then a least-squares refit on its inliers, all relative to the design
centroid. Outliers are only left out of the fit; their residuals are kept. The live monitor is the
*RANSAC monitor* tab of the analysis window (section 12).

**Table** (correction G, SEM → design): Tx, Ty in nm; Mx, My in ppm (= nm per mm of distance);
rotation and orthogonality in degrees. *edge (nm)*: how far that term alone moves the furthest
contact, comparable to the threshold and the residuals.

In [ ]:
contacts = merged[RANSAC_PLACEMENT]
result = ransac_affine(contacts.sem_nm, contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)

print(f"{result.iterations} search iterations; best sample {result.best_sample.tolist()}")
print(f"Inliers: {result.inliers.sum()} / {len(result.inliers)} ({result.inliers.mean():.1%}), "
      f"reference (design centroid): x {result.reference[0] / 1000:.4f} um, y {result.reference[1] / 1000:.4f} um")
display(pd.DataFrame(report_terms(result.model, contacts.design_nm - result.reference),
                     columns=["term (correction G)", "value", "edge (nm)"]).set_index("term (correction G)"))
print("G = [[a, b, tx], [c, d, ty]] about the reference point:")
print(np.array2string(result.model[:2], precision=9, suppress_small=False))
pd.DataFrame({"inliers": error_summary(result.residuals[result.inliers]),
              "all contacts": error_summary(result.residuals)}).round(3)

## 5. Registration error after the RANSAC affine

The final registration error of every contact: its residual after G, G(SEM) − design, outliers
included. The table lists the mean dx, dy of each row of contacts (rows grouped by design y,
`ROW_GAP_NM`), after the affine and, for reference, of the stitched contacts without any affine
(SEM − design). Both are plotted in the *Registration* tabs (section 12).

In [ ]:
registration_error = result.residuals  # (N, 2) G(SEM) - design per merged contact, nm (outliers included)
row_y, row_mean, row_count = row_means(contacts.design_nm[:, 1], registration_error, ROW_GAP_NM)
_, raw_row_mean, _ = row_means(contacts.design_nm[:, 1], contacts.error_nm, ROW_GAP_NM)  # stitched, no affine
print(f"{len(row_y)} rows, {row_count.min()}..{row_count.max()} contacts per row")
pd.DataFrame({"row_y_um": (row_y - result.reference[1]) / 1000, "contacts": row_count,
              "dx_mean_nm": row_mean[:, 0], "dy_mean_nm": row_mean[:, 1],
              "no_affine_dx_mean_nm": raw_row_mean[:, 0], "no_affine_dy_mean_nm": raw_row_mean[:, 1]}).round(3)

## 6. Moving-window affine, compared with the RANSAC affine

A plain least-squares affine in a window `MOVING_WINDOW_UM` tall along y, moved up by
`MOVING_STEP_UM`; each contact is corrected by the window whose centre is nearest to its design y.
The cell compares the summaries and per-row means with the RANSAC result and lists the affine terms
of every window (correction G).

In [ ]:
moving = moving_window_affine(contacts.sem_nm, contacts.design_nm, MOVING_WINDOW_UM * 1000, MOVING_STEP_UM * 1000)
print(f"{len(moving.centres_nm)} windows, {moving.counts.min()}..{moving.counts.max()} contacts per window")
display(pd.DataFrame({"RANSAC inliers": error_summary(result.residuals[result.inliers]),
                      "RANSAC all contacts": error_summary(result.residuals),
                      "moving window all contacts": error_summary(moving.residuals)}).round(3))

window_terms = pd.DataFrame(
    [{label: value for label, value, _ in report_terms(model, contacts.design_nm[moving.window == k] - reference)}
     for k, (model, reference) in enumerate(zip(moving.models, moving.references))])
window_terms.insert(0, "contacts", moving.counts)
window_terms.insert(0, "centre_y_um", (moving.centres_nm - result.reference[1]) / 1000)
window_terms.round(6)

## 7. Row-to-row pitch along y

Rows grouped by design y (`ROW_GAP_NM`); the pitch is the spacing between neighbouring rows' mean y.
Three sets of the same merged contacts: the **design**, the **stitched SEM without any affine** and
the **SEM after the RANSAC affine**. Tile edges (dotted lines in the *Row pitch* tab) are the edges
of each tile's bounding box of contacts.

In [ ]:
pitch_sets = {"stitched, no affine": contacts.sem_nm, "after RANSAC affine": contacts.design_nm + result.residuals}
mid_y, design_pitch = row_pitch(contacts.design_nm[:, 1], contacts.design_nm[:, 1], ROW_GAP_NM)
pitch_table = pd.DataFrame({"row_y_um": (mid_y - result.reference[1]) / 1000, "design_nm": design_pitch})
for name, points in pitch_sets.items():
    pitch_table[f"{name} - design (nm)"] = row_pitch(contacts.design_nm[:, 1], points[:, 1], ROW_GAP_NM)[1] - design_pitch
display(pitch_table.describe().round(3))
worst = pitch_table.iloc[:, 2:].abs().max(axis=1).nlargest(10).index
display(pitch_table.loc[worst].round(3))
tile_edges_y = np.unique(np.round(np.concatenate([centers[:, 1] - sizes[:, 1] / 2, centers[:, 1] + sizes[:, 1] / 2])))

## 8. RANSAC and moving window on the in-image corrected contacts

The RANSAC affine and the moving-window affine of sections 4 and 6, with the same τ, seed, window
and step, on the contacts of section 3b, compared with the translation stitching (the
`RANSAC_PLACEMENT` contacts).

In [ ]:
ic_result = ransac_affine(ic_contacts.sem_nm, ic_contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)
ic_moving = moving_window_affine(ic_contacts.sem_nm, ic_contacts.design_nm, MOVING_WINDOW_UM * 1000,
                                 MOVING_STEP_UM * 1000)
print(f"RANSAC inliers: translation {result.inliers.mean():.1%}, in-image corrected {ic_result.inliers.mean():.1%}")
pd.DataFrame({"translation, RANSAC": error_summary(result.residuals),
              "in-image corrected, RANSAC": error_summary(ic_result.residuals),
              "translation, moving window": error_summary(moving.residuals),
              "in-image corrected, moving window": error_summary(ic_moving.residuals)}).round(3)

## 9. Drift curve along y, removed before the global affine

The mean SEM − design error per row of the translation-stitched contacts (the *no affine* columns of
section 5) follows a straight line where the error is affine; the global affine removes that line.
Drift bends it, and no affine removes a bend. `drift_curve` smooths the row means with a local
straight-line fit `DRIFT_WINDOW_UM` tall (at each row, the least-squares line through the rows within
± half the window, weighted by their contacts); the **bend** is that curve minus the straight line
through all rows. The bend is subtracted from every contact's SEM position (the value of its row) and
the RANSAC affine of section 4 is fitted again. The straight part stays for it, so its terms compare
with section 4. Real mask errors that vary smoothly along y on the scale of the window are removed
too: they cannot be told apart from drift.

Both components are corrected, dx and dy, each as a function of the row's y (nothing depends on x;
the x-dependent terms are left to the global affine). The curve is a local fit, so it has no single
set of coefficients: the first cell prints the window, the straight line left for the global affine
(intercept and slope per component) and, per row, the row mean, the curve and the **bend** that is
subtracted. The second cell applies it and compares the RANSAC inliers and the registration error
(all contacts) of the three sets of contacts, then their RANSAC affine G.

In [ ]:
drift = drift_curve(contacts.design_nm[:, 1], contacts.error_nm, DRIFT_WINDOW_UM * 1000, ROW_GAP_NM)
bend = drift.curve_nm - drift.line_nm  # subtracted from dx and dy, both as a function of the row's y
print(f"Curve: at each of {len(drift.row_y_nm)} rows, the least-squares straight line through the row means within "
      f"+-{drift.window_nm / 2000:g} um of it, taken at that row (a local fit: one value per row, no single polynomial)")
print(f"Straight line through all rows, left for the global affine (y_c = {drift.line_centre_y_nm / 1000:.3f} um):")
for axis, name in enumerate(("dx", "dy")):
    print(f"  {name} = {drift.line_intercept_nm[axis]:+.3f} nm {drift.line_slope[axis] * 1e6:+.3f} ppm x (y - y_c)")
print(f"Bend subtracted (curve - straight line): dx {bend[:, 0].min():+.3f}..{bend[:, 0].max():+.3f} nm, "
      f"dy {bend[:, 1].min():+.3f}..{bend[:, 1].max():+.3f} nm")
pd.DataFrame({"row_y_um": (drift.row_y_nm - result.reference[1]) / 1000,
              "mean_dx_nm": drift.row_mean_nm[:, 0], "curve_dx_nm": drift.curve_nm[:, 0], "bend_dx_nm": bend[:, 0],
              "mean_dy_nm": drift.row_mean_nm[:, 1], "curve_dy_nm": drift.curve_nm[:, 1], "bend_dy_nm": bend[:, 1]}).round(3)

In [ ]:
drift_sem = contacts.sem_nm - drift.correction(contacts.design_nm[:, 1])
drift_result = ransac_affine(drift_sem, contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)

sets = {"translation": (contacts.design_nm, result), "in-image corrected": (ic_contacts.design_nm, ic_result),
        "drift corrected": (contacts.design_nm, drift_result)}
display(pd.DataFrame({name: {"RANSAC inliers (%)": 100 * r.inliers.mean(), **error_summary(r.residuals)}
                      for name, (_, r) in sets.items()}).round(3))
pd.DataFrame({f"G, {name}": {label: value for label, value, _ in report_terms(r.model, design - r.reference)}
              for name, (design, r) in sets.items()})

## 10. Drift per stripe in measurement order, then stitching everything

The images are measured stripe by stripe along y, in a serpentine (`FIRST_STRIPE_UPWARD`: the first
stripe, at the smallest x, from its bottom up, then alternately down and up), so neighbouring stripes
are measured at very different times and can drift differently. The flow, with no affine before the
last step:
1. **Stitch in measurement order** (`tile_stripes`, `measurement_pairs`, `stitch_tiles(only_pairs=...)`):
   only images measured one after the other are tied: the neighbours within each stripe along y and,
   between two stripes, the images where one stripe ends and the next starts (the top images of
   stripes 1/2, the bottom images of stripes 2/3, ...).
2. **Drift per stripe** (`stripe_drift`): for every stripe, the drift curve of its own contacts
   (SEM − design per row, local straight-line fit `DRIFT_WINDOW_UM` tall) for dx and dy; each stripe's
   curve minus the common straight line through the rows of all stripes is subtracted from that
   stripe (the common line is left for the global affine). The curves are in the *Stripe drift
   curves* tab.
3. **Stitch everything** (all overlaps, as in section 2) on the drift-corrected coordinates.
4. **Affine**: RANSAC (section 4) and the moving window (section 6) on the merged contacts.

The first cell lists the stripes and the images tied between them (check that they really were
measured one after the other), the second the drift of every stripe, the third the result next to
the other corrections.

In [ ]:
stripe = tile_stripes(centers, sizes)  # stripe of each tile, 0 = the smallest x
order_pairs = measurement_pairs(centers, stripe, FIRST_STRIPE_UPWARD)
turns = sorted((i, j) for i, j in order_pairs if stripe[i] != stripe[j])
print(f"{stripe.max() + 1} stripes of {np.bincount(stripe).tolist()} tiles; tied between stripes: "
      f"{[(tile_ids[i], tile_ids[j]) for i, j in turns]}")
first_stitch = stitch_tiles(sem_points, centers, sizes, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED, only_pairs=order_pairs)
print(f"Stitched in measurement order: {len(first_stitch.pairs)} of {len(order_pairs)} pairs used, "
      f"{len(first_stitch.unplaced)} tile(s) not stitched")

In [ ]:
first_observations = paired_errors(sem_points, design_points, first_stitch.corrections, design_stitch.corrections)
stripe_drifts = stripe_drift(first_observations.design_nm[:, 1], first_observations.error_nm,
                             stripe[first_observations.tile], DRIFT_WINDOW_UM * 1000, ROW_GAP_NM)
common = stripe_drifts.common
print(f"Common straight line, left for the global affine (y_c = {common.line_centre_y_nm / 1000:.3f} um): "
      + ", ".join(f"{name} = {common.line_intercept_nm[c]:+.3f} nm {common.line_slope[c] * 1e6:+.2f} ppm x (y - y_c)"
                  for c, name in enumerate(("dx", "dy"))))
rows = []
for s, curve in sorted(stripe_drifts.stripes.items()):
    subtracted = curve.curve_nm - stripe_drifts.line(curve.row_y_nm)
    rows.append({"stripe": s + 1, "contacts": int((stripe[first_observations.tile] == s).sum()),
                 "own_slope_dx_ppm": curve.line_slope[0] * 1e6, "own_slope_dy_ppm": curve.line_slope[1] * 1e6,
                 "subtracted_dx_min_nm": subtracted[:, 0].min(), "subtracted_dx_max_nm": subtracted[:, 0].max(),
                 "subtracted_dy_min_nm": subtracted[:, 1].min(), "subtracted_dy_max_nm": subtracted[:, 1].max()})
pd.DataFrame(rows).set_index("stripe").round(3)

In [ ]:
placed = np.nan_to_num(first_stitch.corrections)  # a tile not stitched in measurement order stays nominal
sd_points = [p + placed[k] - stripe_drifts.correction(stripe[k], p[:, 1] + placed[k, 1]) for k, p in enumerate(sem_points)]
sd_stitch = stitch_tiles(sd_points, centers, sizes, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
print(f"Drift-corrected coordinates stitched (all overlaps): {len(sd_stitch.pairs)} pairs used, "
      f"{len(sd_stitch.unplaced)} tile(s) not stitched")
sd_observations = paired_errors(sd_points, design_points, sd_stitch.corrections, design_stitch.corrections)
sd_contacts = merge_observations(sd_observations, design_stitch.corrections, GROUP_RADIUS_NM)
sd_result = ransac_affine(sd_contacts.sem_nm, sd_contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)
sd_moving = moving_window_affine(sd_contacts.sem_nm, sd_contacts.design_nm, MOVING_WINDOW_UM * 1000,
                                 MOVING_STEP_UM * 1000)

sets["stripe drift corrected"] = (sd_contacts.design_nm, sd_result)
display(pd.DataFrame({name: {"RANSAC inliers (%)": 100 * r.inliers.mean(), **error_summary(r.residuals)}
                      for name, (_, r) in sets.items()}).round(3))
display(pd.DataFrame({"stripe drift corrected, moving window": error_summary(sd_moving.residuals)}).round(3))
pd.DataFrame({f"G, {name}": {label: value for label, value, _ in report_terms(r.model, design - r.reference)}
              for name, (design, r) in sets.items()})

## 11. Stitching residuals: translation and translation + rotation, for every input

How well the tiles agree after stitching, for every input: the raw SEM points (section 2), the
in-image corrected points (section 2b), the drift-corrected points (section 10, the final stitch over
all overlaps) and both corrections, in the order they are made (first cell): the in-image corrected
points, their drift per stripe measured and removed as in section 10, then stitched over all overlaps.
Each with two tile models: **translation** (used for all registration results above)
and **translation + rotation** (`stitch_tiles_rigid`, from the same overlaps and tie contacts). Per
stitching (second cell) the RMS tie residual T_i(a) − T_j(b) over the tie contacts of the overlaps within stripes
and between stripes, the largest residual and the number of overlaps (`residual_summary`). A
correction made before stitching that is right should lower these numbers compared with the raw
input. The *Stitching residuals* tab plots them per overlap along y. The global drift curve (section 9)
is applied after stitching and has no stitching of its own.

In [ ]:
ic_first_stitch = stitch_tiles(ic_points, centers, sizes, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED,
                               only_pairs=order_pairs)
ic_first_observations = paired_errors(ic_points, design_points, ic_first_stitch.corrections, design_stitch.corrections)
ic_stripe_drifts = stripe_drift(ic_first_observations.design_nm[:, 1], ic_first_observations.error_nm,
                                stripe[ic_first_observations.tile], DRIFT_WINDOW_UM * 1000, ROW_GAP_NM)
ic_placed = np.nan_to_num(ic_first_stitch.corrections)  # a tile not stitched in measurement order stays nominal
icsd_points = [p + ic_placed[k] - ic_stripe_drifts.correction(stripe[k], p[:, 1] + ic_placed[k, 1])
               for k, p in enumerate(ic_points)]
icsd_stitch = stitch_tiles(icsd_points, centers, sizes, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
print(f"In-image corrected points stitched in measurement order: {len(ic_first_stitch.pairs)} of "
      f"{len(order_pairs)} pairs used, {len(ic_first_stitch.unplaced)} tile(s) not stitched; drift-corrected "
      f"and stitched (all overlaps): {len(icsd_stitch.pairs)} pairs used, {len(icsd_stitch.unplaced)} tile(s) "
      f"not stitched")

In [ ]:
stitchings = {}
for name, points, stitched in (("raw", sem_points, stitch), ("in-image corrected", ic_points, ic_stitch),
                               ("stripe drift corrected", sd_points, sd_stitch),
                               ("in-image + stripe drift corrected", icsd_points, icsd_stitch)):
    stitchings[f"{name}, translation"] = (points, stitched, stitched.corrections)
    stitchings[f"{name}, translation + rotation"] = (points, stitched, stitch_tiles_rigid(points, centers, stitched))
pd.DataFrame({name: residual_summary(*s, centers) for name, s in stitchings.items()}).T.round(3)

### 11b. Both corrections, placed with translation + rotation: RANSAC and moving window

The in-image + stripe drift corrected points of section 11, placed with the **translation +
rotation** of each tile (their `stitch_tiles_rigid` stitching above) instead of translation: errors
against the design (`paired_errors`), merged per contact, then the RANSAC affine (section 4)
and the moving-window affine (section 6) with the same settings, next to the other sets.
A rotation of a tile moves its overlap contacts sideways, so along a stripe the stitching adds up the
small rotations of the tiles into shifts of the tiles in x; the line printed first gives the
rotations and how far the tile centres move against translation. A shift that bends along y is
removed by the moving window but not by the global affine.

In [ ]:
both = "in-image + stripe drift corrected, translation + rotation"  # the set's name in the tables and the window
icsd_rigid = stitchings[both][2]  # (n_tiles, 3, 3) translation + rotation per tile
icsd_rigid_observations = paired_errors(icsd_points, design_points, icsd_rigid, design_stitch.corrections)
icsd_contacts = merge_observations(icsd_rigid_observations, design_stitch.corrections, GROUP_RADIUS_NM)
icsd_result = ransac_affine(icsd_contacts.sem_nm, icsd_contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)
icsd_moving = moving_window_affine(icsd_contacts.sem_nm, icsd_contacts.design_nm, MOVING_WINDOW_UM * 1000,
                                   MOVING_STEP_UM * 1000)

rotation_urad = np.arctan2(icsd_rigid[:, 1, 0], icsd_rigid[:, 0, 0]) * 1e6
moved = np.array([apply_affine(a, c[None])[0] for a, c in zip(icsd_rigid, centers)]) - (centers + icsd_stitch.corrections)
print(f"Rotation per tile: std {np.nanstd(rotation_urad):.1f} urad, {np.nanmin(rotation_urad):+.1f}.."
      f"{np.nanmax(rotation_urad):+.1f} urad; tile centres against translation: x {np.nanmin(moved[:, 0]):+.2f}.."
      f"{np.nanmax(moved[:, 0]):+.2f} nm, y {np.nanmin(moved[:, 1]):+.2f}..{np.nanmax(moved[:, 1]):+.2f} nm")

sets[both] = (icsd_contacts.design_nm, icsd_result)
display(pd.DataFrame({name: {"RANSAC inliers (%)": 100 * r.inliers.mean(), **error_summary(r.residuals)}
                      for name, (_, r) in sets.items()}).round(3))
pd.DataFrame({f"{name}, moving window": error_summary(m.residuals) for name, m in
              (("translation", moving), ("in-image corrected", ic_moving), ("stripe drift corrected", sd_moving),
               (both, icsd_moving))}).round(3)

## 12. Analysis window (closes with the window)

All viewers in one window, one tab each:

- **Stitching**: the viewer of section 2.
- **RANSAC monitor**: press Run or Step to watch the fit of section 4 (same τ and seed).
- **Registration, RANSAC** / **Registration, moving window**: dx and dy error maps (jet, one dot
  per contact), the mean error per row after the affine and, below it, the reference: the mean
  error per row of the stitched contacts **without any affine**. The corrected contacts are extra lines:
  *in-image corrected* (section 8; yellow dx, green dy) and *drift corrected* (section 9; magenta
  dx, cyan dy; RANSAC tab only) and *stripe drift corrected* (section 10; purple dx, lime dy) and *in-image + stripe drift
  corrected, translation + rotation* (section 11b; red dx, blue dy), each with its summary on a line of its own at the top. The **Show** bar above the plots
  switches each set's dx and dy lines on and off (in both row plots; the y axis follows).
- **Moving-window tuner**: sliders for the window height and step; error maps, row means (solid:
  moving window, dashed: RANSAC) and the affine term chosen in the list per window.
- **Row pitch**: section 7, plus every corrected set after its RANSAC affine.
- **In-image corrected** / **Drift corrected**: the *Registration, RANSAC* view of each corrected set:
  dx and dy error maps (same colour scale as *Registration, RANSAC*), the mean error per row after its
  RANSAC affine and, below it, without any affine. *Drift corrected* also shows the bend that was
  subtracted (*drift correction applied*, section 9). *Stripe drift corrected* is the same for
  section 10, *In-image + stripe drift corrected, translation + rotation* for section 11b.
- **Stripe drift curves**: per stripe, the mean dx and dy per row after stitching in measurement order
  (thin) and the drift curve that is subtracted (bold), both minus the common straight line; a check
  box per stripe.
- **Stitching residuals**: section 11, the RMS tie residual per overlap along y for every input and
  tile model (within stripes and between stripes), a check box per stitching.

In [ ]:
app = pg.mkQApp("AffineRANSAC analysis")
stitch_view = PointsStitchView(tile_ids, sem_points, design_points, centers, sizes, stitch, design_stitch,
                               use_opengl=USE_OPENGL)
window = AnalysisWindow(contacts.sem_nm, contacts.design_nm, result, moving, MOVING_WINDOW_UM, MOVING_STEP_UM,
                        RANSAC_THRESHOLD_NM, RANSAC_SEED, RANSAC_DELAY_MS, ROW_GAP_NM, tile_edges_y,
                        stitch_view=stitch_view, use_opengl=USE_OPENGL,
                        extra={"in-image corrected": (ic_contacts.sem_nm, ic_contacts.design_nm, ic_result, ic_moving),
                               "drift corrected": (drift_sem, contacts.design_nm, drift_result, None),
                               "stripe drift corrected": (sd_contacts.sem_nm, sd_contacts.design_nm, sd_result, sd_moving),
                               both: (icsd_contacts.sem_nm, icsd_contacts.design_nm, icsd_result, icsd_moving)})
window.extra_views["drift corrected"].add_rows(contacts.design_nm, drift.correction(contacts.design_nm[:, 1]),
                                               "drift correction applied")  # the bend that was subtracted
window.addTab(StripeDriftView(stripe_drifts, result.reference, use_opengl=USE_OPENGL), "Stripe drift curves")
window.addTab(StitchResidualView(stitchings, centers, result.reference, use_opengl=USE_OPENGL), "Stitching residuals")
window.show()
app.exec()
print(f"Tuner, last setting: window {window.tuner.window.value():g} um, step {window.tuner.step.value():g} um")